# 1.9 · NoSQL 速览 / NoSQL Overview

> **课程定位 / Where this fits**
> **Part 1 第 9 课**——前 8 课全在关系型世界。这一课回答："**什么时候不该用关系型数据库？**" 四大 NoSQL 家族（文档 / KV / 宽列 / 图）各自解决什么问题。
> **Part 1, lesson 9** — the first 8 lessons lived in the relational world. This one answers: "when should you NOT use an RDBMS?" The four NoSQL families and the problems each solves.

> ⚠ **本节没有真的装 MongoDB / Redis**——用 Python + DuckDB 模拟核心概念，**重点是建立选型直觉**，不是学具体 API。具体 API 一天就能上手，**选错数据库返工要一个季度**。
> We don't install MongoDB/Redis here — we simulate the core ideas in Python + DuckDB. The goal is **picking intuition**, not API trivia. APIs take a day to learn; picking the wrong database costs a quarter to undo.

> 💡 **面试相关 / Interview-relevant**
> - "SQL vs NoSQL 怎么选" ★★★★★（系统设计必考）
> - "CAP 定理解释一下" ★★★★
> - "Redis 在你的系统里放哪、存什么" ★★★★（ML 系统设计：特征缓存）
> - "MongoDB 的 schema 灵活是好事还是坏事" ★★★

---

## 学习目标 / Learning Objectives

1. 说出关系型数据库的**三个扩展瓶颈**，以及 NoSQL 用什么换什么。
   Name the three RDBMS scaling bottlenecks and what NoSQL trades away.
2. 用一句话解释 **CAP 定理**并给出 CP / AP 系统各一个例子。
   State CAP in one sentence with a CP and an AP example.
3. 区分**四大 NoSQL 家族**的数据模型、查询方式、代表产品、典型用例。
   Distinguish the four NoSQL families: model, query style, products, use cases.
4. 演示**文档模型**怎么把"JOIN 拍平"成嵌套 JSON，以及代价是什么。
   Show how documents denormalize JOINs into nested JSON — and the cost.
5. 知道 **Redis 在 ML 系统里的三个标准角色**（特征缓存 / 排行榜 / 任务队列）。
   Know Redis's three standard roles in ML systems.
6. 拿到一个新需求，能用**选型决策树**给出数据库建议。
   Use the decision tree to recommend a database for a new requirement.

---

## 目录 / Table of Contents

1. [为什么会有 NoSQL / Why NoSQL Exists](#1)
2. [CAP 定理 ⭐ / The CAP Theorem](#2)
3. [文档数据库（MongoDB）/ Document Stores](#3)
4. [键值数据库（Redis）⭐ / Key-Value Stores](#4)
5. [宽列数据库（Cassandra）/ Wide-Column Stores](#5)
6. [图数据库（Neo4j）/ Graph Databases](#6)
7. [⭐ 选型决策树 / The Decision Tree](#7)
8. [实战：同一份数据的 4 种建模 / One Dataset, Four Models](#8)
9. [小结 / Summary](#9)


<a id="1"></a>
## 1. 为什么会有 NoSQL / Why NoSQL Exists

2000 年代互联网爆炸，关系型数据库撞上三堵墙：
The 2000s web explosion hit three RDBMS walls:

| 瓶颈 / Bottleneck | 关系型的难处 / RDBMS pain | NoSQL 的解法 / NoSQL answer |
|---|---|---|
| **水平扩展** / Scale-out | JOIN + 事务跨机器极难 | 放弃 JOIN，按 key 分片到千台机器 |
| **Schema 演化** / Schema churn | `ALTER TABLE` 大表锁表数小时 | 无固定 schema，文档各自带结构 |
| **写入吞吐** / Write throughput | B-tree 随机写 + ACID 日志开销 | LSM-tree 顺序写 + 最终一致性 |

### 核心交易 / The core trade

> NoSQL 用**放弃（部分）ACID 和 JOIN** 换来 **水平扩展和灵活 schema**。
> NoSQL trades away (some) ACID and JOINs for horizontal scale and flexible schemas.

**不是"NoSQL 更先进"**——是不同的取舍。你的银行余额仍然躺在关系型数据库里，而且应该如此。
This isn't "NoSQL is more advanced" — it's a different trade. Your bank balance still lives in an RDBMS, and it should.

### 命名小注 / Naming note

NoSQL 今天一般解读为 "**Not Only SQL**"。讽刺的是，很多 NoSQL 后来都加回了 SQL 风格查询语言（Cassandra 的 CQL、MongoDB 的聚合管道、甚至 PartiQL）——**SQL 作为查询语言赢了，争的是存储引擎**。
"NoSQL" now reads as "Not Only SQL". Ironically most added SQL-ish query languages back — SQL won as a query language; the fight was about storage engines.


<a id="2"></a>
## 2. CAP 定理 ⭐ / The CAP Theorem

**分布式系统的"不可能三角"**。三选二——而且其中一个不能不选：
The impossible triangle of distributed systems — pick two, and one pick is forced:

| 字母 | 含义 / Meaning |
|---|---|
| **C** onsistency | 任何读都能看到最新写入 / every read sees the latest write |
| **A** vailability | 任何请求都有响应（哪怕数据稍旧）/ every request gets a response |
| **P** artition tolerance | 网络断裂时系统继续工作 / survives network splits |

### 关键洞察 / The key insight

**网络分区一定会发生**（机房之间光缆被挖断是真实事件），所以 **P 必选**。真正的选择只有：
Partitions WILL happen (backhoes cut fiber), so P is mandatory. The real choice:

```
        分区发生时 / When a partition happens:
        ┌─────────────────────────────────────┐
        │  CP: 拒绝部分请求，保证数据一致        │
        │      "宁可不服务，不能给错数据"        │
        │      例: HBase, ZooKeeper, etcd      │
        │      场景: 银行、库存、配置中心        │
        ├─────────────────────────────────────┤
        │  AP: 继续服务，数据可能短暂不一致      │
        │      "宁可给旧数据，不能不服务"        │
        │      例: Cassandra, DynamoDB, Redis  │
        │      场景: 点赞数、购物车、feed 流     │
        └─────────────────────────────────────┘
```

### 最终一致性 / Eventual consistency

AP 系统的承诺："**停止写入后，所有副本终将收敛到同一个值**"。
The AP promise: "stop writing, and all replicas eventually converge."

朋友圈点赞数各人看到的差几秒没人在意——这就是最终一致性的合理场景。**但银行转账绝对不行**。
Nobody cares if like-counts lag by seconds — that's where eventual consistency belongs. Bank transfers: absolutely not.

> 💡 **面试一句话答 / One-liner**:
> "Partitions are inevitable, so CAP is really a choice between consistency and availability **during** a partition. Banks choose C; social feeds choose A."


<a id="3"></a>
## 3. 文档数据库（MongoDB）/ Document Stores

**数据模型**：JSON 文档（MongoDB 实际是 BSON），每个文档自带结构。
**Model**: JSON documents, each carrying its own structure.

### 核心思想：反范式化 / The core idea: denormalization

关系型把"一个订单"拆 3 张表（orders / order_items / products），读的时候 JOIN 回来。
文档型**直接把整个订单存成一个嵌套 JSON**——"**怎么读就怎么存**"。
RDBMS splits an order into 3 tables and JOINs them back. Documents store the whole order as one nested JSON — "store it the way you read it".


In [ ]:
import json
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

# 一个"订单文档"长这样 / What an order document looks like
order_doc = {
    "_id": "ord_001",
    "customer": {"name": "Alice Chen", "country": "US"},     # 嵌套而不是外键 / nested, not FK
    "created": "2026-06-01T10:15:00",
    "items": [                                                # 数组而不是子表 / array, not child table
        {"sku": "TRK-1", "title": "Come Together", "price": 0.99, "qty": 1},
        {"sku": "TRK-9", "title": "So What",        "price": 1.49, "qty": 2},
    ],
    "total": 3.97,
    "tags": ["first_purchase", "promo_jan"],                  # schema 灵活 / flexible schema
}
print(json.dumps(order_doc, indent=2, ensure_ascii=False))


In [ ]:
# MongoDB 的查询长这样（伪代码示意）/ What MongoDB queries look like (illustrative)
mongo_examples = """
# 按嵌套字段查 / Query by nested field
db.orders.find({"customer.country": "US"})

# 数组里任意元素匹配 / Match any array element
db.orders.find({"items.price": {"$gt": 1.0}})

# 聚合管道 = MongoDB 的 GROUP BY / Aggregation pipeline
db.orders.aggregate([
    {"$unwind": "$items"},
    {"$group": {"_id": "$customer.country",
                "revenue": {"$sum": {"$multiply": ["$items.price", "$items.qty"]}}}}
])
"""
print(mongo_examples)


In [ ]:
# DuckDB 也能直接查 JSON——演示"文档查询"的感觉
from pathlib import Path
# DuckDB can query JSON directly — simulating document queries
orders = [
    {"_id": "ord_001", "customer": {"name": "Alice", "country": "US"},
     "items": [{"price": 0.99, "qty": 1}, {"price": 1.49, "qty": 2}]},
    {"_id": "ord_002", "customer": {"name": "Bob", "country": "UK"},
     "items": [{"price": 1.29, "qty": 1}]},
    {"_id": "ord_003", "customer": {"name": "Carol", "country": "US"},
     "items": [{"price": 1.49, "qty": 3}, {"price": 0.50, "qty": 1}]},
]
Path("/tmp/orders.json").write_text("\n".join(json.dumps(o) for o in orders))

conn = duckdb.connect()
print(conn.sql("""
    SELECT
        json_extract_string(j, '$.customer.name')    AS customer,
        json_extract_string(j, '$.customer.country') AS country,
        json_array_length(j, '$.items')               AS n_items
    FROM read_json_objects('/tmp/orders.json') AS t(j)
    WHERE json_extract_string(j, '$.customer.country') = 'US';
""").df())


### 文档模型的优劣 / The trade

| ✅ 赢在 / Wins | ❌ 输在 / Loses |
|---|---|
| 一次读取拿到完整对象（无 JOIN）| **跨文档查询弱**（"买过 X 的人还买过什么"很痛苦）|
| Schema 演化零成本（新字段直接加）| 数据冗余（customer 信息复制进每个订单 → 改名要改 N 处）|
| 天然贴合应用层对象 | 无（跨文档）事务保证（Mongo 4.0+ 有但有限制）|
| 按 `_id` 分片水平扩展容易 | schema 灵活 = "**schema 错误延迟爆炸**"（写时不查，读时才炸）|

> 💡 **面试观点题**："schema 灵活是好事吗？" 好的回答是双面的：**原型期是解放，规模期是债务**——没有 schema 约束意味着每个读取方都要防御性处理字段缺失/类型漂移。
> "Is schema flexibility good?" The strong answer is two-sided: liberating at prototype stage, debt at scale — every reader must defensively handle missing fields and type drift.


<a id="4"></a>
## 4. 键值数据库（Redis）⭐ / Key-Value Stores

**最简单的数据模型**：`key → value`，全内存，微秒级延迟。
The simplest model: key → value, all in RAM, microsecond latency.

| 特性 / Trait | 说明 |
|---|---|
| 全内存 | 比磁盘 DB 快 100-1000× |
| 单线程事件循环 | 无锁竞争，操作天然原子 |
| 丰富 value 类型 | string / list / hash / set / sorted-set / stream |
| TTL 过期 | 每个 key 可设生存时间 → 天然做缓存 |

### Redis 在 ML 系统里的三个标准角色 ⭐

1. **在线特征缓存 / Online feature store**：模型推理时毫秒内取用户特征（Feast 等特征平台的 online store 就是 Redis）
2. **排行榜 / Leaderboards**：sorted-set 天然支持 "top-K + 排名查询"
3. **任务队列 / Task queues**：list 的 LPUSH/BRPOP（Celery 的 broker 之一）

下面用纯 Python 模拟这三种用法的**语义**：
Below we simulate the semantics of all three in pure Python:


In [ ]:
import time
import heapq

# 模拟 1: 带 TTL 的特征缓存 / Feature cache with TTL
class MiniRedis:
    def __init__(self):
        self._data = {}      # key -> (value, expire_ts or None)

    def setex(self, key, ttl_seconds, value):
        self._data[key] = (value, time.time() + ttl_seconds)

    def get(self, key):
        if key not in self._data:
            return None                      # cache miss
        value, expire = self._data[key]
        if expire and time.time() > expire:
            del self._data[key]              # 过期即删 / lazy expiry
            return None
        return value

cache = MiniRedis()

def get_user_features(user_id):
    # 真实系统：先查缓存，miss 才去算/查数仓
    # Real systems: cache first; on miss compute / hit the warehouse
    key = f"features:user:{user_id}"
    feats = cache.get(key)
    if feats is not None:
        return feats, "HIT"
    feats = {"avg_spend_30d": 42.5, "n_orders": 7}   # 假装是昂贵计算 / pretend-expensive
    cache.setex(key, ttl_seconds=300, value=feats)    # 缓存 5 分钟 / cache 5 min
    return feats, "MISS"

print(get_user_features(1))    # 第一次 MISS / first: miss
print(get_user_features(1))    # 第二次 HIT  / second: hit


In [ ]:
# 模拟 2: sorted-set 排行榜 / Sorted-set leaderboard
# Redis 命令: ZADD leaderboard 1500 "alice" / ZREVRANGE leaderboard 0 2
leaderboard = {}                                  # member -> score

def zadd(member, score): leaderboard[member] = score
def ztop(k): return sorted(leaderboard.items(), key=lambda kv: -kv[1])[:k]

zadd("alice", 1500); zadd("bob", 2300); zadd("carol", 1800); zadd("dave", 900)
print("Top-3:", ztop(3))

# 模拟 3: list 任务队列 / List-based task queue
# Redis 命令: LPUSH jobs '{"task": ...}' / BRPOP jobs
from collections import deque
jobs = deque()
jobs.appendleft('{"task": "retrain_model", "priority": "high"}')   # LPUSH
jobs.appendleft('{"task": "refresh_features"}')
print("worker takes:", jobs.pop())                                  # BRPOP（先进先出）


### Redis 的边界 / Where Redis stops

- **不是主存储**：内存贵 + 持久化（RDB/AOF）是"尽力而为"——掉电可能丢最后几秒
- **单 key 值上限 512 MB**，但实践 > 1 MB 就该反思设计
- 复杂查询（多条件过滤）不是它的菜——那是 SQL / 文档库的事

> 💡 **ML 系统设计面试**：被问到"线上推理延迟高"时，"**把热特征放 Redis、TTL 5 分钟、miss 回源特征平台**"是标准答案组件。
> In ML system design interviews, "hot features in Redis with a 5-min TTL, fall back to the feature store on miss" is a standard answer component.


<a id="5"></a>
## 5. 宽列数据库（Cassandra）/ Wide-Column Stores

**为"海量写入 + 按 key 查询"而生**。代表：Cassandra、HBase、Bigtable（Google 论文鼻祖）。
Built for massive writes + by-key reads. Cassandra, HBase, Bigtable.

### 数据模型 / The model

```
分区键 (partition key)  →  决定数据在哪台机器
聚簇键 (clustering key) →  决定分区内怎么排序

(user_id) → [ (ts1, event1), (ts2, event2), ... ]   ← 一个分区 = 一个用户的全部事件，按时间排好
```

### 为什么写入快 / Why writes are fast

**LSM-tree**（Log-Structured Merge tree）：写入先进内存 memtable + 顺序追加 commit log，**永远不随机写磁盘**；后台再合并成 SSTable。
LSM-trees buffer writes in memory + append-only log — no random disk writes ever; background compaction merges later.

| | B-tree (RDBMS) | LSM-tree (Cassandra) |
|---|---|---|
| 写 | 随机 I/O，需就地更新 | **顺序 I/O，吞吐极高** |
| 读 | 一次定位 | 可能查多个 SSTable（用布隆过滤器加速）|
| 适合 | 读多写少 | **写多读少**（IoT / 日志 / 消息）|

### 致命约束 / The fatal constraint ⭐

**查询必须带分区键**。`WHERE user_id = 42 AND ts > ...` ✅；`WHERE event_type = 'click'`（不带 user_id）❌ 全集群扫描，等于自杀。
Queries MUST include the partition key. Without it = full-cluster scan = suicide.

所以 Cassandra 的设计哲学是"**先想清楚查询，再设计表**"（query-first design）——和关系型"先建模、再随便查"刚好相反。**同一份数据按不同查询路径建多张表**是常态。
Hence "query-first design" — the opposite of relational modeling. Duplicating data across multiple query-shaped tables is normal practice.

**典型用户**：Netflix（观看历史）、Discord（消息存储，万亿级）、Uber（行程事件）。


<a id="6"></a>
## 6. 图数据库（Neo4j）/ Graph Databases

**数据模型**：节点 + 边 + 属性。**为"多跳关系查询"而生**。
Nodes + edges + properties. Built for multi-hop relationship queries.

### 什么时候 SQL 会跪 / Where SQL kneels

"找 Alice 的朋友的朋友的朋友"（3 跳）：
- SQL：3 层自连接，每层全表 JOIN，**指数爆炸**
- 图数据库：从 Alice 节点**沿边指针走 3 步**，只触碰相关节点

$$T_{\text{SQL}} = O(\text{JOIN}^{hops}) \qquad T_{\text{graph}} = O(\text{邻居数}^{hops}\text{，从起点局部展开})$$

### Cypher 查询语言（Neo4j）

```cypher
// 找 Alice 的 2 跳内朋友，按共同好友数排序
MATCH (a:Person {name: 'Alice'})-[:FRIEND*1..2]-(fof:Person)
WHERE fof <> a
RETURN fof.name, count(*) AS strength
ORDER BY strength DESC
```

**ASCII 箭头画图模式** `(节点)-[:关系]->(节点)` —— 可读性吊打等价 SQL。
The ASCII-art pattern syntax beats the equivalent SQL for readability.


In [ ]:
# 用 Python 字典模拟"沿边行走 vs JOIN"的差别
# Simulate edge-walking vs JOIN
friends = {
    "alice": ["bob", "carol"],
    "bob":   ["alice", "dave", "erin"],
    "carol": ["alice", "frank"],
    "dave":  ["bob"],
    "erin":  ["bob", "frank"],
    "frank": ["carol", "erin", "grace"],
    "grace": ["frank"],
}

def friends_within(start, hops):
    # BFS：图数据库内部本质就是这个 / BFS — what a graph DB does natively
    seen, frontier = {start}, {start}
    for _ in range(hops):
        frontier = {f for person in frontier for f in friends[person]} - seen
        seen |= frontier
    return sorted(seen - {start})

print("Alice 1 跳:", friends_within("alice", 1))
print("Alice 2 跳:", friends_within("alice", 2))
print("Alice 3 跳:", friends_within("alice", 3))


**典型用例**：欺诈环检测（"这 5 个账号共用 2 个设备和 1 张银行卡"）、推荐（协同关系）、知识图谱（Part 16 会深入 + GNN）。
Use cases: fraud-ring detection, recommendations, knowledge graphs (Part 16 goes deep with GNNs).

> 💡 **何时不用图库**：关系只有 1-2 跳、或主要是聚合统计 → SQL 足够。图库赢在 **≥3 跳 + 不定长路径**。
> Skip graph DBs when relationships are 1-2 hops or the workload is aggregation — SQL suffices. Graphs win at ≥3 hops and variable-length paths.


<a id="7"></a>
## 7. ⭐ 选型决策树 / The Decision Tree

```
新需求来了，存哪？
│
├── 需要强事务（钱 / 库存 / 订单状态）？
│     └── ✅ → PostgreSQL / MySQL（别犹豫）
│
├── 是分析查询（聚合 / 报表 / OLAP）？
│     └── ✅ → DuckDB（本地）/ Snowflake / BigQuery（云数仓，1.10 节）
│
├── 毫秒级 KV 读取（缓存 / 会话 / 特征）？
│     └── ✅ → Redis
│
├── 对象结构嵌套深 + schema 频繁变 + 按 ID 取整个对象？
│     └── ✅ → MongoDB（或 Postgres JSONB——见下方注）
│
├── 写入洪水（IoT / 日志 / 消息）+ 查询永远带 key？
│     └── ✅ → Cassandra / ScyllaDB
│
├── 多跳关系查询（欺诈环 / 社交 / 知识图谱）？
│     └── ✅ → Neo4j / NebulaGraph
│
└── 全文搜索 / 日志检索？
      └── ✅ → Elasticsearch / OpenSearch
```

### ⭐ 工业潜规则："先 Postgres，不行再说"

**PostgreSQL 的 JSONB + GIN 索引能覆盖 80% 的"我需要 MongoDB"场景**；加上 Redis 缓存能覆盖大部分"我需要 NoSQL"的冲动。
Postgres JSONB + GIN covers ~80% of "I need MongoDB" cases; add Redis caching and most NoSQL urges dissolve.

> "Use Postgres until you can prove you can't." —— 多数资深架构师的默认立场。引入每种新数据库 = 多一套运维 / 备份 / 监控 / 招聘要求。
> Every additional database = another ops/backup/monitoring/hiring burden.


<a id="8"></a>
## 8. 实战：同一份数据的 4 种建模 / One Dataset, Four Models

**"用户给歌曲点赞"** 这一个业务，分别用 4 种模型建模——看清各自的形状和擅长的查询。
One feature — "users like tracks" — modeled four ways, exposing what each shape is good at.


In [ ]:
# ============ 模型 1: 关系型 / Relational ============
# 规范化三表：likes 是关联表 / Normalized: likes as junction table
relational = """
users(user_id PK, name)
tracks(track_id PK, title)
likes(user_id FK, track_id FK, liked_at)        -- 复合主键 (user_id, track_id)

-- 擅长: 任意方向的查询都行
SELECT t.title, COUNT(*) FROM likes l JOIN tracks t USING (track_id)
GROUP BY t.title ORDER BY 2 DESC;                -- 最受欢迎的歌
"""
print(relational)


In [ ]:
# ============ 模型 2: 文档 / Document ============
# 按"读取单位"反范式化：用户文档内嵌点赞列表
doc = {
    "_id": "user_42",
    "name": "Alice",
    "likes": [
        {"track": "Come Together", "at": "2026-06-01"},
        {"track": "So What",       "at": "2026-06-03"},
    ],
}
print(json.dumps(doc, indent=2))
print("""
擅长: "打开 Alice 的主页" → 1 次读取全拿到
痛点: "谁点赞了 So What?" → 扫所有用户文档（要么再建反向集合 = 双写）
""")


In [ ]:
# ============ 模型 3: 键值 / Key-Value (Redis) ============
kv = """
SET   user:42:likes        → {"Come Together", "So What"}      (set 类型)
SADD  track:so_what:likers → {42, 17, 99}                       (反向 set，双写)
INCR  track:so_what:count  → 3                                  (计数器)

擅长: 点赞/取消点赞 O(1)；计数器实时读
痛点: "上周点赞趋势" 这类分析 → 导出到数仓再算
"""
print(kv)


In [ ]:
# ============ 模型 4: 图 / Graph ============
graph = """
(Alice:User)-[:LIKED {at: '2026-06-01'}]->(ct:Track {title: 'Come Together'})
(Alice:User)-[:LIKED]->(sw:Track {title: 'So What'})
(Bob:User)-[:LIKED]->(sw)

擅长: "喜欢 Alice 喜欢的歌的人还喜欢什么?" → 协同过滤一条 Cypher:
MATCH (me:User {name:'Alice'})-[:LIKED]->(t)<-[:LIKED]-(other)-[:LIKED]->(rec)
WHERE NOT (me)-[:LIKED]->(rec)
RETURN rec.title, count(*) ORDER BY count(*) DESC

痛点: 简单计数聚合反而绕；运维生态比 SQL 弱
"""
print(graph)


### 结论 / The takeaway

| 查询 / Query | 最舒服的模型 / Most natural model |
|---|---|
| "Alice 的主页（她的全部点赞）" | 文档 / KV |
| "最受欢迎的 100 首歌" | 关系型 / 数仓 |
| "实时点赞计数" | KV (Redis INCR) |
| "喜欢相似歌曲的用户" （协同过滤）| 图 |

**同一份数据没有"最优模型"，只有"对当前查询最优的模型"**——大型系统经常 4 种并存（写主库 + CDC 同步到各专用存储）。
There is no single best model — only the best model for each query. Big systems often run all four, synced via CDC from the write-side primary.


<a id="9"></a>
## 9. 小结 / Summary

### 四大家族一页表 / The four families on one page

| 家族 / Family | 代表 | 数据模型 | 杀手锏 | 致命伤 | CAP 倾向 |
|---|---|---|---|---|---|
| 文档 / Document | MongoDB | 嵌套 JSON | 读写整对象、schema 灵活 | 跨文档查询/事务弱 | 可调（默认偏 CP）|
| 键值 / KV | Redis | key → 富类型 value | 微秒延迟、原子操作 | 不是主存储、容量=内存 | AP |
| 宽列 / Wide-column | Cassandra | 分区键 + 聚簇键 | 写入洪水、线性扩容 | 查询必须带分区键 | AP（可调）|
| 图 / Graph | Neo4j | 节点 + 边 | ≥3 跳关系查询 | 聚合分析绕、生态小 | 多为 CP |

### 💡 面试速查 / Interview must-knows

1. **CAP 一句话**：分区必然发生，真正选的是"分区期间要 C 还是要 A"
2. **SQL vs NoSQL 不是先进 vs 落后**：是 ACID+JOIN vs 扩展性+灵活 schema 的交易
3. **Redis 在 ML 系统的三角色**：特征缓存 / 排行榜 / 任务队列
4. **Cassandra 铁律**：查询必须带分区键 → query-first 建模
5. **图库的甜点**：≥3 跳 + 不定长路径（欺诈环）
6. **默认立场**："先 Postgres（+JSONB+Redis），证明不行再引入新库"

### 下一节预告 / Next up

**Part 1.10 · 数据仓库** —— OLTP vs OLAP 正式分家：星型模型、事实表/维度表、缓慢变化维（SCD）。DS 日常查的表 90% 长这个样子。
**Part 1.10 · Data Warehouse** — star schemas, fact/dim tables, slowly changing dimensions. 90% of the tables a DS queries daily look like this.
